# 02 – 10-K text analysis

For every company in the universe and every fiscal year since `start_year`:

1. **Download the 10-K** (primary HTML document) from SEC EDGAR
2. **Extract two sections** – *Item 1A Risk Factors* and *Item 7 MD&A*
3. **Dictionary sentiment** – share of Loughran-McDonald negative / positive / uncertainty / litigious / constraining words, net tone
4. **Year-over-year change** – cosine & Jaccard similarity to last year's section, share of new sentences

⏱️ **First run is slow** (~800 filings, roughly 30–60 min: each 10-K is 1–10 MB of HTML).
Only the extracted sections are cached (`data/interim/sections/`, gzip), so an interrupted run
continues where it stopped and later runs take seconds.

**Why these signals?**
* *Loughran & McDonald (2011)* showed generic sentiment lists misclassify finance words ("liability", "tax"); their 10-K-based dictionary does not.
* *Cohen, Malloy & Nguyen – "Lazy Prices" (2020)*: firms that substantially change their 10-K text subsequently underperform – markets are slow to read long documents. Change is a stronger signal than tone.

In [ ]:
from IPython.display import display, HTML
import pandas as pd

from stockpred.config import load_settings
from stockpred.edgar import EdgarClient
from stockpred.text import build_text_features, load_section, new_sentences

pd.set_option("display.float_format", "{:,.3f}".format)
settings = load_settings()
client = EdgarClient(settings.sec_user_agent, settings.raw_dir / "edgar",
                     settings.max_requests_per_second, settings.max_age_days)
SECTIONS_DIR = settings.data_dir / "interim" / "sections"
universe = pd.read_csv(settings.processed_dir / "universe.csv")

## 1. Build text features

In [ ]:
result = build_text_features(client, universe, SECTIONS_DIR, start_year=settings.start_year,
                             predecessors=settings.cik_predecessors)
text = result.features
text.to_parquet(settings.processed_dir / "text_features.parquet", index=False)
result.filings.to_parquet(settings.processed_dir / "text_filings.parquet", index=False)
print(text.shape)

## 2. Extraction quality

Share of 10-Ks where the section was found. Section extraction relies on headings like *"Item 1A. Risk Factors"*;
companies with a cross-reference layout (banks, ExxonMobil…) are handled by a title-based fallback for MD&A.
Anything well below 100 % deserves a manual look at one of the filings.

In [ ]:
quality = (text.pivot_table(index="ticker", columns="section", values="extracted", aggfunc="mean")
               .sort_values("risk_factors"))
print(f"Overall extraction rate: {text['extracted'].mean():.1%}")
display(quality.head(15).style.format("{:.0%}").background_gradient(cmap="RdYlGn", vmin=0, vmax=1))

## 3. Cross-section: who changed their Risk Factors the most in the latest year?

Low Jaccard similarity / high share of new sentences = substantial rewrite.

In [ ]:
latest = (text[(text["section"] == "risk_factors") & text["extracted"]]
          .sort_values("fiscal_year").groupby("ticker").tail(1))
cols = ["ticker", "fiscal_year", "share_new_sentences", "sim_jaccard", "length_change", "net_tone", "lm_uncertainty"]
display(latest[cols].sort_values("share_new_sentences", ascending=False).head(15))

## 4. Read the new risks

The numbers say *that* something changed – the new sentences say *what*. Pick a ticker:

In [ ]:
TICKER = latest.sort_values("share_new_sentences", ascending=False).iloc[0]["ticker"]   # most changed
SECTION = "risk_factors"

f = result.filings[result.filings["ticker"] == TICKER].sort_values("fiscal_year")
cur, prev = f.iloc[-1], f.iloc[-2]
added = new_sentences(load_section(SECTIONS_DIR, cur.cik, cur.accession, SECTION),
                      load_section(SECTIONS_DIR, prev.cik, prev.accession, SECTION))
print(f"{TICKER}: {len(added)} new sentences in FY{cur.fiscal_year} vs FY{prev.fiscal_year}\n")
for s in added[:20]:
    print("•", s)

### Limitations
* Dictionary methods ignore context and negation ("not material" counts as negative). FinBERT (optional, `requirements-nlp.txt`) can be added later.
* Risk Factors are mostly boilerplate → tone barely moves; changes over time are more informative than levels.
* Section boundaries are heuristic; very unusual layouts may be missed (see section 2).